In [4]:
# ================== CLEAN SETUP ==================
import os, warnings, logging, re
os.environ["WANDB_DISABLED"] = "true"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# ================== IMPORTS ==================
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from torch.nn import CrossEntropyLoss

# ================== TEXT CLEANING ==================
def clean_text(text):
    text = re.sub(r"http\S+", "", text)   # remove URLs
    text = re.sub(r"@\w+", "", text)      # remove mentions
    text = re.sub(r"#", "", text)         # remove hashtags
    text = re.sub(r"\s+", " ", text)      # remove extra spaces
    return text.strip().lower()

# ================== LOAD DATA ==================
df = pd.read_csv("dataset_1_2.csv")

# Map labels if they are strings
if df["label"].dtype == "object":
    label_mapping = {"Non-Hate": 0, "Hate": 1}
    df["label"] = df["label"].map(label_mapping)

df["label"] = df["label"].astype(int)
df["text"] = df["text"].apply(clean_text)

# Split dataset
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

# ================== CLASS WEIGHTS ==================
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# ================== TOKENIZER ==================
model_name = "microsoft/deberta-v3-base"   # 🔥 best DeBERTa balance of speed & accuracy
tokenizer = AutoTokenizer.from_pretrained(model_name)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = TextDataset(train_texts, train_labels)
test_dataset  = TextDataset(test_texts, test_labels)

# ================== MODEL ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
).to(device)

# ================== CUSTOM TRAINER WITH WEIGHTED LOSS ==================
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=class_weights.to(device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# ================== TRAINING ARGS ==================
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,           # 🔥 tuned lower than BERT
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=6,           # 🔥 longer training
    weight_decay=0.01,
    warmup_ratio=0.1,             # warmup helps stability
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # 🔥 stop if no progress
)

# ================== TRAIN ==================
trainer.train()

# ================== EVALUATE ==================
predictions = trainer.predict(test_dataset)
y_pred = predictions.predictions.argmax(axis=-1)

print("\n✅ Test Accuracy:", accuracy_score(test_labels, y_pred))
print("\n✅ Classification Report:\n", classification_report(test_labels, y_pred, target_names=["Non-Hate", "Hate"]))

# ================== INFERENCE ==================
test_dataset_nolabels = TextDataset(test_texts, labels=None)
predictions_nolabels = trainer.predict(test_dataset_nolabels)
final_preds = predictions_nolabels.predictions.argmax(axis=-1)

print("\n🔮 Predicted labels (0=Non-Hate, 1=Hate):", final_preds[:50])

Using device: cuda
{'loss': 0.6254, 'grad_norm': 7.8504838943481445, 'learning_rate': 1.854581673306773e-05, 'epoch': 1.0}
{'eval_loss': 0.5203892588615417, 'eval_accuracy': 0.7574123989218329, 'eval_runtime': 6.8419, 'eval_samples_per_second': 108.45, 'eval_steps_per_second': 6.869, 'epoch': 1.0}
{'loss': 0.4507, 'grad_norm': 3.6988418102264404, 'learning_rate': 1.4840637450199205e-05, 'epoch': 2.0}
{'eval_loss': 0.5032891631126404, 'eval_accuracy': 0.7789757412398922, 'eval_runtime': 6.5925, 'eval_samples_per_second': 112.553, 'eval_steps_per_second': 7.129, 'epoch': 2.0}
{'loss': 0.3127, 'grad_norm': 2.519705057144165, 'learning_rate': 1.1135458167330676e-05, 'epoch': 3.0}
{'eval_loss': 0.5474917888641357, 'eval_accuracy': 0.7816711590296496, 'eval_runtime': 6.6002, 'eval_samples_per_second': 112.42, 'eval_steps_per_second': 7.121, 'epoch': 3.0}
{'loss': 0.2098, 'grad_norm': 2.107194185256958, 'learning_rate': 7.430278884462152e-06, 'epoch': 4.0}
{'eval_loss': 0.7044529914855957, 'e